# Explorer les données complètes — Fuel Analyser
Sources brutes intactes, instantané à 47 colonnes, historique complet en une table par année.
Exécute les cellules dans l’ordre. Aucun téléchargement ni écriture de données n’est déclenché.
Les colonnes JSON contiennent toutes les informations sources ; les prix suspects et doublons restent conservés et signalés.


In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/cleaned").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvre ce notebook dans le projet extrait avec ses données.")
HISTORICAL = ROOT / "data/cleaned/historical"


## 1. Instantané complet : les 47 colonnes
Les colonnes conservent leur nom et leur ordre. Le JSON est stocké en texte. Les dates fournies avec un fuseau sont converties en UTC ; les identifiants restent du texte.

In [2]:
current = pd.read_parquet(ROOT / "data/cleaned/current.parquet")
print(current.shape)
display(pd.DataFrame({"colonne": current.columns, "type": current.dtypes.astype(str).values}))
display(current.head())
current_quality = json.loads((ROOT / "data/quality/current.json").read_text(encoding="utf-8"))
display(pd.DataFrame(current_quality["issues"]).head(20))


(9827, 47)


,colonne,type
0,id,string
1,latitude,Float64
2,longitude,Float64
3,Code postal,string
4,pop,string
5,Adresse,string
6,Ville,string
7,horaires,string
8,services,string
9,prix,string


,id,latitude,longitude,Code postal,pop,Adresse,Ville,horaires,services,prix,...,Carburants indisponibles,Carburants en rupture temporaire,Carburants en rupture definitive,Automate 24-24 (oui/non),Services proposés,Département,code_departement,Région,code_region,horaires détaillés
0,27340001,49.301637,1.165554,27340,R,75 Avenue Forêt de Bord,Les Damps,"{""@automate-24-24"":""1"",""jour"":[{""@id"":""1"",""@no...","{""service"":[""Automate CB 24/24"",""Lavage automa...","[{""@nom"":""Gazole"",""@id"":""1"",""@maj"":""2026-11-06...",...,"SP95,GPLc",<NA>,SP95;GPLc,True,"Automate CB 24/24,Lavage automatique,Relais co...",Eure,27,Normandie,28,"Automate-24-24, Lundi00.00-23.59, Mardi00.00-2..."
1,33114001,44.623,-0.787,33114,R,Avenue du Médoc ZAC Eyrialis,Le Barp,"{""@automate-24-24"":"""",""jour"":[{""@id"":""1"",""@nom...","{""service"":[""Boutique alimentaire"",""Vente de g...","[{""@nom"":""Gazole"",""@id"":""1"",""@maj"":""2026-11-06...",...,"SP95,E85,GPLc",<NA>,SP95;E85;GPLc,False,"Boutique alimentaire,Vente de gaz domestique (...",Gironde,33,Nouvelle-Aquitaine,75,"Lundi01.00-01.00, Mardi01.00-01.00, Mercredi01..."
2,34150003,43.651,3.547,34150,R,Route de Pézenas,Gignac,"{""@automate-24-24"":""1"",""jour"":[{""@id"":""1"",""@no...","{""service"":[""Bar"",""Boutique alimentaire"",""Lava...","[{""@nom"":""Gazole"",""@id"":""1"",""@maj"":""2026-11-10...",...,"SP95,GPLc",<NA>,SP95;GPLc,True,"Bar,Boutique alimentaire,Lavage automatique,Lo...",Hérault,34,Occitanie,76,"Automate-24-24, Lundi01.00-01.00, Mardi, Mercr..."
3,92230008,48.9327,2.3044,92230,R,192 Avenue Louis Roche,Gennevilliers,<NA>,<NA>,"[{""@nom"":""Gazole"",""@id"":""1"",""@maj"":""2026-11-06...",...,"E85,GPLc",<NA>,E85;GPLc,False,<NA>,Hauts-de-Seine,92,Île-de-France,11,<NA>
4,25115001,47.257413,5.924087,25115,R,Z.A. les Salines,Pouilley-les-Vignes,"{""@automate-24-24"":""1"",""jour"":[{""@id"":""1"",""@no...","{""service"":[""Laverie"",""DAB (Distributeur autom...","[{""@nom"":""Gazole"",""@id"":""1"",""@maj"":""2026-11-09...",...,SP95,<NA>,SP95,True,"Laverie,DAB (Distributeur automatique de bille...",Doubs,25,Bourgogne-Franche-Comté,27,Automate-24-24


""


## 2. Inventaire des années
La table conserve chaque élément de prix, même invalide, ainsi qu’une ligne par station sans prix. Les doublons ne sont pas supprimés : le total des lignes n’est pas un nombre d’observations uniques.

In [3]:
reports = [json.loads(p.read_text(encoding="utf-8")) for p in sorted((ROOT / "data/quality").glob("[0-9][0-9][0-9][0-9].json"))]
quality = pd.DataFrame(reports).set_index("year")
display(quality[["station_count", "source_price_elements", "retained_rows", "no_price", "invalid_price", "invalid_date", "duplicate_price", "conflicting_price"]])


,station_count,source_price_elements,retained_rows,no_price,invalid_price,invalid_date,duplicate_price,conflicting_price
year,,,,,,,,
2007,7904,866993,866993,0,548,548,0,4
2008,8394,1422159,1422159,0,605,605,0,6
2009,9387,1398296,1398299,3,1229,1229,0,192
2010,10130,1445254,1445261,7,1718,1718,0,0
2011,10001,1613150,1613160,10,1338,1338,0,2
2012,10256,1904467,1904476,9,1229,1229,0,2
2013,10807,2152004,2152010,6,1415,1415,0,312
2014,11064,2199274,2199281,7,1326,1326,0,14
2015,12333,3307860,3307874,14,2601,2601,6312,2576


## 3. Charger une année
Une ligne = un élément de déclaration pour un carburant, ou une station sans déclaration (`no_price=True`). Les caractéristiques de station sont celles fournies dans l’archive annuelle : elles ne sont pas garanties à la date de chaque prix. Pour limiter la mémoire, le chargement initial ne lit pas les longs JSON ; toutes les colonnes restent disponibles dans le même fichier.

In [ ]:
YEAR = 2022
FUEL = "Gazole"  # Gazole, SP95, SP98, E10, E85, GPLc ou Tous
file = HISTORICAL / f"{YEAR}.parquet"
filters = None if FUEL == "Tous" else [("fuel", "==", FUEL)]
columns = ["station_id", "postal_code", "city", "address", "road_type", "latitude", "longitude", "fuel", "fuel_code", "price_eur_l", "price_reported_at", "price_raw", "date_raw", "archive_year", "source_archive", "source_station_index", "source_price_index", "no_price", "invalid_price", "invalid_date", "unknown_fuel", "suspect_price", "outside_archive_year", "duplicate_price", "conflicting_price"]
prices = pd.read_parquet(file, columns=columns, filters=filters)
print(f"{len(prices):,} lignes chargées")
display(prices.head())
display(prices.isna().sum().rename("Valeurs manquantes"))


## 4. Voir toutes les colonnes pour une station
Les services, horaires, anciennes ouvertures, ruptures et fermetures sont conservés en JSON. `station_source_json` contient tous les attributs et éléments de la station hors prix ; `price_source_json` conserve le prix original. Une liste vide ou une valeur absente ne prouve pas l’absence d’un service dans la réalité.

In [ ]:
STATION_ID = str(prices["station_id"].dropna().iloc[0]) if prices["station_id"].notna().any() else None
station_rows = pd.read_parquet(file, filters=[("station_id", "==", STATION_ID)]) if STATION_ID else pd.DataFrame()
display(station_rows.head())
if not station_rows.empty:
    for column in ["hours_json", "opening_json", "services_json", "outages_json", "closures_json"]:
        print(column)
        value = station_rows.iloc[0][column]
        display(json.loads(value) if pd.notna(value) else None)


## 5. Construire une vue analytique sans effacer les données
Les règles ci-dessous sont propres à l’analyse : elles ne modifient pas le fichier nettoyé. Les lignes invalides, prix atypiques, dates hors année et conflits sont écartés de cet exemple. Les répétitions exactes sont dédupliquées uniquement en mémoire.

In [ ]:
from src.processing.quality_rules import assess_prices, analytical_events
assessed = assess_prices(prices)
usable = assessed.loc[assessed['analysis_eligible']].copy()
display(assessed.filter(like='dq_').sum().rename('Observations signalees'))
display(usable.groupby('fuel')['price_eur_l'].agg(['count', 'min', 'median', 'max']))


## 6. Évolution des déclarations
La médiane précédente décrit les déclarations. Pour la courbe, une station compte au maximum une fois par jour et par carburant. Aucun prix n’est prolongé sur les jours suivants. La couverture varie : il ne s’agit pas d’un panel constant ni d’une moyenne pondérée par les ventes.

In [ ]:
events = analytical_events(prices)
daily_prices = events.assign(day=events["price_reported_at"].dt.normalize()).sort_values("price_reported_at").drop_duplicates(["station_id", "fuel", "day"], keep="last")
# Si la dernière déclaration du jour est invalide, ne pas la remplacer par une plus ancienne.
daily_prices = daily_prices.loc[daily_prices["analysis_eligible"]]
daily = daily_prices.groupby(["day", "fuel"]).agg(price=("price_eur_l", "mean"), stations=("station_id", "nunique")).reset_index()
if not daily.empty:
    daily.pivot(index="day", columns="fuel", values="price").plot(figsize=(12, 4), title=f"Prix moyens des stations déclarantes — {YEAR}")
    plt.ylabel("€/L"); plt.tight_layout(); plt.show()
    daily.pivot(index="day", columns="fuel", values="stations").plot(figsize=(12, 3), title="Couverture quotidienne")
    plt.tight_layout(); plt.show()
else:
    print("Aucune observation exploitable pour la sélection.")


## Tes analyses
Question, sélection, règle de calcul, résultat, limites et conclusion. Les fichiers bruts et les JSON permettent de vérifier les transformations. Ne concatène pas l’instantané avec les archives sans définir les clés et le traitement du recouvrement temporel.

In [ ]:
# Écris tes analyses ici.
